# M03 · Getting seismic data: STP, stations, time windows, responses

**Week 2 · about 5.7 h (+ 1 h optional)** · before this: M01, M02, M04

> **한국어 요약** 개인 STP 계정으로 상주 지진 여섯 개의 파형을 받고, 관측망·관측소·채널 코드와 UTC 시간 창을 읽는 법을 배웁니다. 사건 당일 운영 중이던 관측소를 고르고, 받은 자료를 세고, 안병석(부경대학교)의 StationMetadata로 계기 감도를 계산합니다. 마지막으로 자료가 하나도 없는 동료의 시간 창을 바로잡습니다.

**You will**

- Read the network, station and channel codes of a record
- Write record windows in UTC and find the stations running then
- Count what arrived and compute a channel's sensitivity
- Find out why a time window holds no data

**New words:** STP (파형 전송 프로그램), channel (채널), time window (시간 창, 시간 윈도우), station epoch (관측소 운영 기간, 에포크), counts (카운트, 디지털 값), instrument response (계기 응답), overall sensitivity (전체 감도), envelope (포락선) · [all terms](../../../docs/glossary.md)

**Optional reading:** Ahn, Kang & Jung (2024), English abstract and figures; Havskov & Alguacil (2016), instrument correction

Log any AI help in `work/ai_log/M03.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M03.yaml](../../../.sgtl/criteria/M03.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"              # the STP dataset of this module
EVENT_ID = "20190721020418"         # event folder = UTC origin time, YYYYMMDDhhmmss (the ML 3.9 event)
EPICENTER = (36.50, 128.10)         # latitude, longitude of the Sangju earthquakes (KMA), degrees
DEPTH_KM = 14.0                     # focal depth of the ML 3.9 event (KMA), km
RADIUS_KM = 100.0                   # stations within this epicentral distance (km)
BEFORE_S, AFTER_S = 30.0, 470.0     # record window: seconds before and after the origin
DEMO_STATION = "KS.SAJB"            # Part 6: station of the response-removal demo
PRE_FILT = (0.3, 0.6, 30.0, 40.0)   # Part 6: pre-filter corners (Hz) of the response removal
TRAP_ROW = 1                        # Part 7: the row of event_catalog.csv that the colleague used
FIGURE_DIR = "figures"              # your figures go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M03")

import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from obspy import UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth
from obspy.signal.filter import envelope

DATA = st.data.root() / DATASET                 # this module's dataset, outside the repository
SAC_ROOT = DATA / "stp_download" / "SAC"        # one folder per event, one folder per band inside it
META = st.data.root() / "metadata"              # StationMetadata, installed by: bin/sgtl-install metadata --apply
Path(FIGURE_DIR).mkdir(exist_ok=True)


def need(path, what):
    """Stop with a plain message when a file or folder that a cell needs is missing; return it as a Path."""
    if not Path(path).exists():
        raise FileNotFoundError(f"{what} is missing ({path}). Part 1 shows how to get it.")
    return Path(path)

## Part 1 · Your STP account and this module's data (about 25 min)

**STP** is the program that fetches Korean **seismograms**, records of ground motion, from the data center. Your
mentor gives you an STP account and the client `stp-client.pl`. Keep both in the hidden folder `~/.sgtl/`,
outside the repository:

1. `mkdir -p ~/.sgtl && chmod 700 ~/.sgtl`
2. Copy the client to `~/.sgtl/stp-client.pl`.
3. Write two lines into `~/.sgtl/credentials.env`: `STP_USER=<your user name>` and `STP_PASS=<your password>`.
4. `chmod 600 ~/.sgtl/credentials.env ~/.sgtl/stp-client.pl` (only you may read them)

The cell checks both files without reading them.

In [ ]:
private_dir = Path.home() / ".sgtl"
for name in ("credentials.env", "stp-client.pl"):
    item = private_dir / name
    if not item.is_file():
        print(f"~/.sgtl/{name}: not found. Follow the steps above when you have your account.")
        continue
    mode = item.stat().st_mode & 0o777
    verdict = "only you can read it: good" if mode & 0o077 == 0 else "OTHER USERS CAN READ IT: run chmod 600 on it"
    print(f"~/.sgtl/{name}: found, permissions {mode:03o}, {verdict}")

**Rules without exceptions.** Whoever has these files can download data in your name.

- Never commit them or copy them into the repository.
- Never paste them, or other people's data or paths, into a chat, an e-mail or an AI tool.
- Never lend your account. If someone saw your password, tell your mentor the same day.

**Fetch the data.** In a server terminal, run `bin/sgtl-install tools metadata`, read its plan, then run it again
with `--apply`. It installs PocketQuake and the station metadata. Then:

1. `sgtl data fetch sangju2019`: a **dry run**, the plan only;
2. `sgtl data fetch sangju2019 --apply`: the download;
3. `sgtl data verify sangju2019`: checks the files.

The data go to your data folder, never into the repository. Tasks 1 and 2 work without them. This dataset:

In [ ]:
manifest = yaml.safe_load(st.data.repo_path("data", "manifests", "stp_datasets.yaml").read_text(encoding="utf-8"))
entry = manifest["datasets"][DATASET]
print(entry["title"], "| networks", entry["networks"], "| stations within", entry["station_radius_km"], "km",
      "| window", entry["window_s"], "s around each origin")
pd.DataFrame(entry["events"])

**The files.** `event_catalog/event_catalog.csv` lists the six events, `station_table/` every station STP knows,
and `stp_download/stp_batch.txt` the requests. Each `win` line asks for one window.
`win KS DUSB HH_ 2018/02/24,21:16:07.000000 500s` asks for all HH channels of KS.DUSB, 500 s from 21:16:07 UTC. The
records are **SAC files** in `stp_download/SAC/<event>/<band>/`, one per channel; event folders are named after the
UTC origin.

In [ ]:
if not SAC_ROOT.is_dir():
    print(f"No waveforms yet in {DATA}. Fetch them first (above); tasks 1 and 2 work without them.")
else:
    for event_dir in sorted(p for p in SAC_ROOT.iterdir() if p.is_dir()):
        per_band = {b.name: len(list(b.glob("*.sac"))) for b in sorted(event_dir.iterdir()) if b.is_dir()}
        print(event_dir.name, per_band)
    batch_lines = (DATA / "stp_download" / "stp_batch.txt").read_text(encoding="utf-8").splitlines()
    print(len(batch_lines), "lines in stp_batch.txt; the first three:")
    print("\n".join(batch_lines[:3]))

## Part 2 · Networks, stations and channels (about 25 min)

Every record has a **SEED id** of four codes, such as `KS.SAJB..HHZ`:

- network: `KS` is the Korea Meteorological Administration (KMA), `KG` the Korea Institute of Geoscience and
  Mineral Resources (KIGAM);
- station: three to five letters or digits;
- location: empty in this dataset (hence the two dots);
- **channel**: three letters, explained below.

In [ ]:
event_dir = need(SAC_ROOT / EVENT_ID, f"The event folder {EVENT_ID}")
names = pd.DataFrame([f.name.split(".") for f in sorted(event_dir.glob("*/*.sac"))],
                     columns=["start", "network", "station", "channel", "suffix"])
print(f"{len(names)} files in event {EVENT_ID}; files per channel code:")
print(names["channel"].value_counts().sort_index().to_dict())
print("stations per network:", names.groupby("network")["station"].nunique().to_dict())
headers = read(str(event_dir / "*" / "*KG.CHS.*.sac"), headonly=True)
for tr in headers:
    print(tr.id, f"{tr.stats.sampling_rate:g} samples/s, {tr.stats.npts} samples, starts {tr.stats.starttime}")

The three letters of a channel code:

1. **band**: `H` and `E` record 80-250 samples per second, `B` and `S` 10-80. `H` and `B` come from a
   **broadband seismometer**, `E` and `S` from a **short-period** one;
2. **instrument**: `H` is a high-gain and `L` a low-gain seismometer. In KS and KG, `G` is an **accelerometer**,
   which stays on scale in strong shaking;
3. **component**: `Z` vertical, `N` north, `E` east.

So `HHZ` is a broadband vertical channel and `HGZ` an accelerometer's.

**Look at the records** in SeisComP's `scrttv` with the [SeisComP GUI workshop](../../../docs/seiscomp_gui.md) (its
rehearsal first). Find the four codes of one stream and its time span in UTC. You do not need the SAC program.

**Two station lists.** `st.data.stations()` lists today's stations. The STP **roster** (`station_table/`) also
lists stations that stopped recording, **retired stations**. A station that moves a short distance often gets a
new code, such as `CHJ2` after `CHJ`.

In [ ]:
today = st.data.stations()
roster = pd.concat([pd.read_csv(need(DATA / "station_table" / f"{net}_station.csv", "The STP station roster"))
                    for net in ("KS", "KG")], ignore_index=True)
roster = roster.rename(columns={"Network": "network", "Code": "station", "Latitude": "latitude",
                                "Longitude": "longitude", "Elevation": "elevation_m"})
in_today = set(today["network"] + "." + today["station"])
in_roster = set(roster["network"] + "." + roster["station"])
print(f"today's list: {len(in_today)} stations; STP roster: {len(in_roster)} stations")
print(f"in the roster only: {len(in_roster - in_today)}; in today's list only: {len(in_today - in_roster)}")
print("a few roster-only codes:", sorted(in_roster - in_today)[:12])
twins = roster.groupby("station")["network"].nunique()
print("station codes used by both networks:", sorted(twins[twins > 1].index))

**Watch out:** a station code alone does not name a station. `CHS` and `BRD` exist in both networks, at
different sites: always keep the network with it (`KG.CHS`).

## Part 3 · Time windows in UTC (about 10 min)

Each record is cut in a **time window** around the origin time: here 30 s before (noise) to 470 s after (the
waves). Data centers, file names and SAC headers use **UTC**. Korean reports, and this dataset's
`event_catalog.csv`, use **KST**, UTC + 9 h. Compare the two:

In [ ]:
kst_rows = pd.read_csv(need(DATA / "event_catalog" / "event_catalog.csv", "The dataset's event catalog"))
display(kst_rows)
print("event folders (UTC):", sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir()))
kma = st.data.sequence(DATASET)
print(kma.loc[kma["event_id"].isin([e["event_id"] for e in entry["events"]]),
              ["event_id", "origin_utc", "origin_kst", "ml"]].to_string(index=False))

A time without a zone is **naive**; a time with its zone is **aware** and names one moment. The KMA list of the
course (KMA catalog, KOGL Type 1) keeps both clocks.

**Watch out:** the rows of `event_catalog.csv` are naive. Nothing in them says that 11:04:18 is Korean time.

### M03-T1 · Time windows in UTC · core · about 35 min

Write `utc_window(origin_utc, before_s, after_s)`. It returns `(start, end)`, two ISO 8601 strings in UTC with
the zone written out (`Z` or `+00:00`). `start` is `before_s` seconds before the origin, `end` is `after_s`
seconds after it. The origin can come in many spellings; the docstring lists them.

Example: `utc_window("2019-07-21T11:04:18+09:00", 30, 470)` gives
`("2019-07-21T02:03:48+00:00", "2019-07-21T02:12:08+00:00")`.

In [ ]:
# Before you code: the first row of event_catalog.csv says 2018-02-25 06:16:37 KST. When, in UTC, does its record
# window (30 s before, 470 s after) start and end?
st.predict("M03-T1", """

""")

In [ ]:
def utc_window(origin_utc, before_s, after_s):
    """Start and end of a record window around an origin time, as UTC text.

    origin_utc : origin time in any of these spellings: text with Z or an offset ("2019-07-21T02:04:18Z",
                 "2019-07-21T11:04:18+09:00"), text without a zone, a pandas Timestamp with or without a zone, a
                 Python datetime with a zone, or an ObsPy UTCDateTime (always UTC). With a zone or an offset it
                 is that moment; without one it is UTC.
    before_s   : float, seconds before the origin (>= 0, fractions allowed)
    after_s    : float, seconds after the origin (>= 0, fractions allowed)
    returns    : (start, end), two ISO 8601 strings in UTC with the zone written out (Z or +00:00), correct to
                 the millisecond: start is before_s seconds before the origin, end is after_s seconds after it
    """
    # Step 1: turn origin_utc into one moment in UTC, whatever its spelling (an ObsPy UTCDateTime included).
    # Step 2: move back before_s seconds and forward after_s seconds.
    # Step 3: write both moments as ISO text that keeps the zone.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T1: write your code here")

In [ ]:
st.check("M03-T1", func=utc_window)

Your function on the six events, compared with the requests in `stp_batch.txt`:

In [ ]:
sent = {line.split()[4] for line in (DATA / "stp_download" / "stp_batch.txt").read_text().splitlines()
        if line.startswith("win ")}
for ev in entry["events"]:
    start, end = utc_window(ev["origin_utc"], BEFORE_S, AFTER_S)
    as_stp = pd.Timestamp(start).strftime("%Y/%m/%d,%H:%M:%S.%f")
    length = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds()
    print(f"{ev['event_id']}  ML {ev['ml']}  {start} to {end} ({length:.0f} s)  STP start {as_stp}  "
          f"sent: {as_stp in sent}")
first = kst_rows.iloc[0].astype(int)
written = f"{first.Year}-{first.Month:02d}-{first.Day:02d} {first.Hour:02d}:{first.Minute:02d}:{first.Second:02d}"
start, end = utc_window(written, BEFORE_S, AFTER_S)
print(f"the first catalog row, {written}, read as if it were UTC: {start} to {end}")

**Explain:** The last line reads the first catalog row as if it were UTC. What would STP send back for that
window?

## Part 4 · Which stations were running? (about 15 min)

A station's life is split into **station epochs**: periods with the same site and instruments. A new sensor or a
move starts a new epoch. Each epoch has a start and an end date; an end in 2099 or 9999 means still running. The
cell keeps each station's first start and last end.

In [ ]:
def station_epochs(roster_df, center, half_width_deg, xml_dir):
    """One row per roster station near `center` that has a StationXML file: first start and last end date.

    roster_df      : DataFrame with network and station columns (the STP roster)
    center         : (latitude, longitude) in degrees
    half_width_deg : keep stations within this many degrees of the center in latitude and in longitude
    xml_dir        : folder of NET.STA.xml files
    returns        : DataFrame network, station, latitude, longitude, start_date, end_date (dates without a time
                     zone; end_date NaT = still running), and the list of stations without a file
    """
    lat0, lon0 = center
    near = roster_df[(roster_df["latitude"] - lat0).abs().le(half_width_deg)
                     & (roster_df["longitude"] - lon0).abs().le(half_width_deg)]
    rows, no_file = [], []
    for net, sta in zip(near["network"], near["station"]):
        path = Path(xml_dir) / f"{net}.{sta}.xml"
        if not path.is_file():
            no_file.append(f"{net}.{sta}")
            continue
        site = read_inventory(str(path))[0][0]
        starts = [c.start_date for c in site.channels]
        ends = [c.end_date for c in site.channels]
        last = None if any(e is None or e.year >= 2099 for e in ends) else max(ends)
        rows.append({"network": net, "station": sta, "latitude": site.latitude, "longitude": site.longitude,
                     "start_date": pd.Timestamp(min(starts).date),
                     "end_date": pd.NaT if last is None else pd.Timestamp(last.date)})
    return pd.DataFrame(rows), no_file


epochs, without_file = station_epochs(roster, EPICENTER, 1.2, need(META / "stationxml", "StationMetadata"))
print(f"{len(epochs)} stations with metadata near Sangju; {len(without_file)} roster stations have no file")
print("with an open end (none, 2099 or 9999):", int(epochs["end_date"].isna().sum()))
epochs.head()

### M03-T2 · Stations running on a given day · core · about 45 min

Write `stations_within(stations_df, lat, lon, radius_km, on_date)`. It returns the rows of `stations_df` running
on `on_date` within `radius_km` of (`lat`, `lon`), plus a column `distance_km`, nearest first. The distance is the
**epicentral distance**, along the surface (M01).

A row is running when it started on or before that day and has no end date or ended on or after it.

Example: on 2019-07-21, a station that ended on 2019-07-21 is running; one that ended on 2019-07-20 is not.

In [ ]:
# Before you code: for the ML 3.9 event of 2019-07-21, will today's station list or the metadata epochs give more
# stations within 100 km?
st.predict("M03-T2", """

""")

In [ ]:
def stations_within(stations_df, lat, lon, radius_km, on_date):
    """Station epochs running on a given day within a radius of a point, nearest first.

    stations_df : DataFrame, one row per station epoch (one station can have several rows, for example before
                  and after a move), with network, station, latitude, longitude (degrees), start_date and usually
                  end_date (pandas dates without a zone), maybe other columns. A missing end date (NaT), or no
                  end_date column at all, means still running.
    lat, lon    : float, the center in degrees
    radius_km   : float, the largest distance in km
    on_date     : the day, as text "YYYY-MM-DD" or a pandas Timestamp without a zone
    returns     : DataFrame of the rows running on on_date (start_date <= day, and end_date missing or >= day:
                  the first and the last day both count) whose great-circle distance from (lat, lon) is at most
                  radius_km (sphere of radius 6371 km; a geodesic library is fine too). All columns of
                  stations_df plus distance_km, sorted by distance_km, nearest first; an empty DataFrame with
                  those columns when no row qualifies. stations_df itself is not changed.
    """
    # Step 1: which rows are running on that day (both ends count; no end date means still running)?
    # Step 2: the distance of every row from the center, in km.
    # Step 3: keep the rows that pass both tests, add distance_km, sort.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T2: write your code here")

In [ ]:
st.check("M03-T2", func=stations_within)

Your function on both lists, for the day of the ML 3.9 event:

In [ ]:
event_day = pd.Timestamp(EVENT_ID[:8])
lat0, lon0 = EPICENTER
by_today = stations_within(today, lat0, lon0, RADIUS_KM, event_day)
by_epochs = stations_within(epochs, lat0, lon0, RADIUS_KM, event_day)
with_data = set(names["network"] + "." + names["station"])
for label, table in (("today's list", by_today), ("metadata epochs", by_epochs)):
    ids = set(table["network"] + "." + table["station"])
    print(f"{label}: {len(ids)} stations running within {RADIUS_KM:g} km; {len(ids & with_data)} of them sent data;"
          f" {len(with_data - ids)} stations with data are not on it: {sorted(with_data - ids)}")
print(f"{len(with_data)} stations sent data for {EVENT_ID}")

The map shows the stations that sent records and those that did not. Two codes can share one site, so symbols
may overlap.

In [ ]:
import pygmt

region = [lon0 - 1.35, lon0 + 1.35, lat0 - 1.05, lat0 + 1.05]
quiet = by_epochs[~(by_epochs["network"] + "." + by_epochs["station"]).isin(with_data)]
sent_rows = roster[(roster["network"] + "." + roster["station"]).isin(with_data)]
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig_map = pygmt.Figure()
    fig_map.basemap(region=region, projection="M10c", frame=["af", f"+tStations of event {EVENT_ID}"])
    fig_map.coast(land="gray92", water="white", shorelines="0.25p,gray40")
    fig_map.plot(data=np.array([[lon0, lat0, 2.0 * RADIUS_KM]]), style="E-k", pen="0.6p,gray30,-")  # 100-km circle
    fig_map.plot(data=quiet[["longitude", "latitude"]].to_numpy(), style="i0.22c", pen="0.6p,red3",
                 label="Listed in metadata; no record+S0.2c")
    fig_map.plot(data=sent_rows[["longitude", "latitude"]].to_numpy(), style="t0.22c", fill="dodgerblue",
                 pen="0.3p,black", label="Record received+S0.2c")
    fig_map.plot(data=np.array([[lon0, lat0]]), style="a0.45c", fill="gold", pen="0.5p,black", label="Epicenter+S0.3c")
    fig_map.legend(position="jBR+o0.2c", box="+gwhite+p0.5p")
    fig_map.basemap(map_scale=st.plot.scale_bar_spec(region, km=50))
fig_map.savefig(Path(FIGURE_DIR) / "m03_stations_map.png", dpi=200)
fig_map.show()

**Explain:** Which stations that sent data are missing from today's list, and why?

## Part 5 · Did everything arrive? (about 10 min)

A **completeness check** compares what you asked for with what you got. When a station has no data, STP sends
nothing and says nothing. Count before you compute: a station with only one or two of its three components can
break a later step.

### M03-T3 · What arrived? · core · about 45 min

Write `completeness_table(sac_root)` for a SAC tree like yours (one folder per event, one per band inside it). It
returns `(counts, gaps)`:

- `counts`: the number of stations per event (rows) and band (columns), 0 where there are none;
- `gaps`: the stations that have some, but not all, of the components E, N and Z.

Count a station as `NET.STA`: one station code can exist in two networks.

In [ ]:
# Before you code: from the file counts of Part 1, how many HH stations do you expect for the ML 3.9 event?
st.predict("M03-T3", """

""")

In [ ]:
def completeness_table(sac_root):
    """Stations per event and band in a SAC tree, and the stations that miss a component.

    sac_root : str or Path: one folder per event (named by its UTC origin time), inside it one folder per band
               (HH, HG, EL, ...), SAC files named <start>.<NET>.<STA>.<CHA>.sac. Folders can be empty or missing
               and other files can lie anywhere: count only files ending in .sac that lie directly in a band
               folder, and read the codes from their names. Change nothing in the tree.
    returns  : (counts, gaps)
               counts: DataFrame; index = the event folder names (text, sorted); columns = every band folder name
                       found anywhere (sorted); values = stations (NET.STA) with at least one SAC file there,
                       0 when none (whole numbers, no NaN)
               gaps:   DataFrame with the columns event, station ("NET.STA"), band, missing: one row per event,
                       station and band with some but not all of E, N, Z (last letter of the channel code);
                       missing = the absent letters in alphabetical order ("N", "EZ"); sorted by event, band,
                       station; empty, with these four columns, when nothing is missing
    """
    # Step 1: walk the event folders and their band folders; collect, per event and band, the components found
    #         for every network-and-station.
    # Step 2: fill the counts table (every event folder a row, every band folder a column, 0 where nothing).
    # Step 3: list the stations whose components are not all three, with the absent letters.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T3: write your code here")

In [ ]:
st.check("M03-T3", func=completeness_table)

Your function on the real tree, and the requests per event and band:

In [ ]:
counts, gaps = completeness_table(SAC_ROOT)
requested = {}
for line in (DATA / "stp_download" / "stp_batch.txt").read_text().splitlines():
    if line.startswith("dir "):
        folder = Path(line.split(maxsplit=1)[1])
        key = (folder.parent.name, folder.name)
    elif line.startswith("win "):
        requested[key] = requested.get(key, 0) + 1
asked = pd.Series(requested).unstack().reindex(index=counts.index, columns=counts.columns)
print("stations that sent data, per event and band:")
print(counts.to_string())
print("\nshare of the requested stations that sent data:")
print((counts / asked).round(2).to_string())
print(f"\n{len(gaps)} event-station-band combinations miss a component")
display(gaps.head())

**Explain:** Why is the share of requests that returned data so far below 1, even for the ML 3.9 event?

## Part 6 · Instrument responses (about 25 min)

A **digitizer** measures the sensor's voltage and writes **counts**, whole numbers. The **instrument response**
says how the recording chain turns ground motion (m/s, or m/s² for an accelerometer) into counts.

The responses come from **StationMetadata**, integrated by **Byeong-Seok Ahn (Pukyong National University)** from
the records of KMA, KIGAM and the Korea Institute of Nuclear Safety. Cite it as the file below says:

In [ ]:
acknowledgment = need(META / "ACKNOWLEDGMENT.txt", "StationMetadata (run: bin/sgtl-install metadata --apply)")
print("\n".join(acknowledgment.read_text(encoding="utf-8").splitlines()[:14]))

The response is a list of **stages**, each with a **gain**:

1. the sensor, for example 1500 V per m/s;
2. the digitizer, for example 419,430 counts per V;
3. digital filters, usually 1.

The product of all gains is the **overall sensitivity**, in counts per m/s (or per m/s²). One station's epochs
(`B` channels record 20 samples per second):

In [ ]:
chs = read_inventory(str(META / "stationxml" / "KG.CHS.xml"))
epoch_rows = []
for channel in chs[0][0].channels:
    if channel.code.endswith("Z"):
        epoch_rows.append({"channel": channel.code, "start": str(channel.start_date)[:19],
                           "end": str(channel.end_date)[:19], "sensor": channel.sensor.description,
                           "samples/s": channel.sample_rate,
                           "stage gains": [s.stage_gain for s in channel.response.response_stages],
                           "units": channel.response.response_stages[0].input_units,
                           "stored sensitivity": channel.response.instrument_sensitivity.value})
pd.DataFrame(epoch_rows)

### M03-T4 · Counts per meter per second · core · about 45 min

Write `sensitivity_at(inventory, seed_id, time)`. It finds the epoch of channel `seed_id` that covers `time`. It
returns `(value, ground_units)`: the size of the product of the stage gains, and the input units of the first
stage. With no epoch at `time`, it returns `None`.

Example: stage gains −2000, 1 and 500,000 with input units `"M/S"` give `(1e9, "M/S")`.

Not allowed: the stored `instrument_sensitivity` (not even printed) or `get_response`.

In [ ]:
# Before you code: from the table above, about how many counts per m/s do you expect for KG.CHS..HHZ in 2019?
st.predict("M03-T4", """

""")

In [ ]:
def sensitivity_at(inventory, seed_id, time):
    """Overall sensitivity of one channel at one moment, from the gains of its response stages.

    inventory : ObsPy Inventory (from read_inventory) with one or more networks, stations, channels and channel
                epochs. Each channel epoch has .code, .location_code, .start_date, .end_date (None when it has not
                ended) and .response.response_stages, a list of stages with .stage_gain, .input_units and
                .output_units.
    seed_id   : text "NET.STA.LOC.CHA"; all four codes must match, and LOC may be empty ("KG.CHS..HHZ")
    time      : ObsPy UTCDateTime. An epoch covers it when start_date <= time and (end_date is None or
                time <= end_date).
    returns   : (value, ground_units): value > 0, the size of the product of the gains of all stages of the
                covering epoch, in counts per unit of ground motion; ground_units the input units of the first
                stage as written in the file (e.g. "M/S" or "M/S**2"). None when no epoch of that channel covers
                time. Work it out from the stages: do not read instrument_sensitivity or call get_response.
    """
    # Step 1: split the id into its four codes.
    # Step 2: find the channel epoch with those four codes that covers time.
    # Step 3: multiply the gains of all its stages; keep the size; read the units of the first stage.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T4: write your code here")

In [ ]:
st.check("M03-T4", func=sensitivity_at)

Your function on every vertical channel of the ML 3.9 event. `negative first gain` counts minus signs in the first
stage:

In [ ]:
origin = UTCDateTime(f"{EVENT_ID[:8]}T{EVENT_ID[8:]}")
z_rows = []
for path in sorted(event_dir.glob("*/*Z.sac")):
    _, net, sta, cha, _ = path.name.split(".")
    inv = read_inventory(str(META / "stationxml" / f"{net}.{sta}.xml"))
    found = sensitivity_at(inv, f"{net}.{sta}..{cha}", origin)
    first_gain = None
    if found is not None:
        first_gain = inv.select(channel=cha, time=origin)[0][0][0].response.response_stages[0].stage_gain
    z_rows.append({"id": f"{net}.{sta}..{cha}", "counts per unit": None if found is None else found[0],
                   "units": None if found is None else found[1],
                   "negative first gain": first_gain is not None and first_gain < 0})
sens = pd.DataFrame(z_rows)
print(sens.groupby(sens["id"].str[-3:-1]).agg(**{"channels": ("id", "size"),
                                                 "negative first gain": ("negative first gain", "sum"),
                                                 "median": ("counts per unit", "median")}).to_string())
print("units:", sens["units"].value_counts().to_dict())

**Removing the response.** ObsPy's `remove_response` divides the record's spectrum by the response, frequency by
frequency. A **pre-filter** (`PRE_FILT`) fades out the frequencies where the response is tiny, so the division
does not blow up. Compare it with the counts divided by your one number:

In [ ]:
net, sta = DEMO_STATION.split(".")
raw = read(str(event_dir / "HH" / f"*.{net}.{sta}.HHZ.sac"))[0]
station_xml = read_inventory(str(META / "stationxml" / f"{net}.{sta}.xml"))
value, units = sensitivity_at(station_xml, raw.id, raw.stats.starttime)
rough = (raw.data - raw.data.mean()) / value
velocity = raw.copy()
velocity.detrend("demean")
velocity.taper(0.05)
velocity.remove_response(inventory=station_xml, output="VEL", pre_filt=PRE_FILT)
seconds = raw.times(reftime=origin)
fig, ax = plt.subplots(figsize=(8.0, 3.2))
ax.plot(seconds, rough * 1e6, lw=0.6, color="0.55", label="Counts divided by the sensitivity")
ax.plot(seconds, velocity.data * 1e6, lw=0.6, color="tab:blue", label="Response removed")
ax.set_xlim(-5, 40)
ax.set_xlabel("Seconds after the origin (UTC)")
ax.set_ylabel("Ground velocity (µm/s)")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "m03_response_removal.png", dpi=150)
print(f"{raw.id}: {value:.4g} counts per {units}; peak |velocity| {np.abs(velocity.data).max() * 1e6:.1f} µm/s "
      f"(response removed) and {np.abs(rough).max() * 1e6:.1f} µm/s (counts / sensitivity)")
print("correlation of the two traces:", round(float(np.corrcoef(rough, velocity.data)[0, 1]), 3))

KS.SAJB is one of the 11 broadband channels with a negative first gain. By the StationXML convention, that means
"records upside down", but the stored sensitivity is positive. `remove_response` warns and uses the stored value,
so the trace keeps the sign of the counts. We checked this event: all 11 move first like their neighbors and like
the accelerometer at the same site.

**Watch out:** here a negative stage gain is not a reliable polarity flag. Take first-motion signs (M20, M51)
from the counts; test a channel that looks reversed on several earthquakes before you flip it.

**Is the amplitude sensible?** The IASPEI **local magnitude** formula uses the largest horizontal swing $A$ (nm)
of a simulated **Wood-Anderson seismograph**. Here $R$ is the **hypocentral distance** (km), from the hypocenter
to the station:

$$M_L = \log_{10} A + 1.11 \log_{10} R + 0.00189\,R - 2.09$$

Made for California, it should agree within a few tenths. A unit slip of 1000 in $A$ would move ML by 3.

In [ ]:
WOOD_ANDERSON = {"poles": [-6.283 + 4.712j, -6.283 - 4.712j], "zeros": [0j, 0j], "gain": 1.0, "sensitivity": 1.0}
ml_rows = []
for north_file in sorted(event_dir.glob("HH/*HHN.sac")):
    pair = read(str(north_file)) + read(str(north_file).replace("HHN.sac", "HHE.sac"))
    net, sta = pair[0].stats.network, pair[0].stats.station
    station_xml = read_inventory(str(META / "stationxml" / f"{net}.{sta}.xml"))
    peaks = []
    for tr in pair:
        tr.detrend("demean")
        tr.taper(0.05)
        tr.remove_response(inventory=station_xml, output="DISP", pre_filt=PRE_FILT)
        tr.simulate(paz_simulate=WOOD_ANDERSON)
        peaks.append(np.abs(tr.data).max() * 1e9)          # nm
    epi_km = gps2dist_azimuth(lat0, lon0, pair[0].stats.sac.stla, pair[0].stats.sac.stlo)[0] / 1000.0
    hypo_km = float(np.hypot(epi_km, DEPTH_KM))
    ml_rows.append({"station": f"{net}.{sta}", "epi_km": epi_km, "amplitude_nm": max(peaks),
                    "ml": np.log10(max(peaks)) + 1.11 * np.log10(hypo_km) + 0.00189 * hypo_km - 2.09})
ml_table = pd.DataFrame(ml_rows).sort_values("epi_km", ignore_index=True)
catalog_ml = kma.loc[kma["origin_utc"] == pd.Timestamp(origin.datetime, tz="UTC"), "ml"].iloc[0]
print(ml_table.round(2).to_string())
print(f"median of {len(ml_table)} stations: ML {ml_table['ml'].median():.2f} "
      f"(spread {ml_table['ml'].quantile(0.25):.2f}-{ml_table['ml'].quantile(0.75):.2f}); KMA catalog: ML {catalog_ml}")

**Explain:** Why does `sensitivity_at` need the event time? Use the KG.CHS table in your answer.

## Part 7 · A window without an earthquake (about 10 min)

A colleague wants a figure of the largest Sangju earthquake at every broadband station, 10 s before to 60 s after
the origin. The colleague takes the origin time from row `TRAP_ROW` of the catalog file and the event folder with
the same row number. No record holds data in the window, and no error appears. "The broadband stations missed it,"
says the colleague. Can that be true?

Your evidence: the **envelope** of each record, a smooth curve that follows the size of the wiggles
(`obspy.signal.filter.envelope`). Its peak over the noise is a **signal-to-noise ratio**. It usually peaks on the
S wave, about 1 s after the origin for every 3.5 km of distance.

### M03-T5 · A window without an earthquake · core · about 60 min

Run the colleague's cell below and find the mistake. Then write two functions (see their docstrings):

- `corrected_window(row, before_s, after_s)`: like `utc_window`, the window around the true origin of one row of
  `event_catalog.csv`;
- `envelope_evidence(records, row, origin, wrong_origin, before_s, after_s)`: a table with, for every record, its
  envelope peak in your window and its samples in the colleague's.

Then build the table `evidence` with them, in the third code cell.

In [ ]:
# The colleague's cell. Run it; do not edit it.
trap_row = kst_rows.iloc[TRAP_ROW]
origin = UTCDateTime(int(trap_row["Year"]), int(trap_row["Month"]), int(trap_row["Day"]), int(trap_row["Hour"]),
                     int(trap_row["Minute"]), int(trap_row["Second"]))
trap_dir = sorted(p for p in SAC_ROOT.iterdir() if p.is_dir())[TRAP_ROW]
records = read(str(trap_dir / "HH" / "*.HHZ.sac"))
window = records.slice(origin - 10.0, origin + 60.0)
print(f"event {trap_dir.name}: origin {origin}, ML {trap_row['Magnitude']}")
print(f"{len(records)} broadband vertical records; {len(window)} of them hold data from 10 s before to 60 s after")
print("Conclusion: the broadband stations did not record this earthquake.")

In [ ]:
# Before you code: when (in UTC) do the records in trap_dir start and end, and where does the colleague's
# window fall?
st.predict("M03-T5", """

""")

In [ ]:
def corrected_window(row, before_s, after_s):
    """Time window around the origin of one row of event_catalog.csv, in UTC.

    row      : pandas Series or dict with Year, Month, Day, Hour, Minute, Second and maybe other fields, written
               like the rows of event_catalog.csv (numbers, maybe floats such as 2019.0; Second may have a
               fraction), at any date of the KMA list (1978 to today)
    before_s : float, seconds before the origin (>= 0)
    after_s  : float, seconds after the origin (>= 0)
    returns  : (start, end) as in utc_window: two ISO 8601 strings in UTC with the zone written out, correct to
               the millisecond, for the window around the moment of the earthquake that the row describes (its
               origin time in the KMA catalog)
    """
    # Step 1: build the clock reading of the row, and say which clock it was read on.
    # Step 2: turn it into the window in UTC (your task 1 can help).
    # YOUR CODE HERE
    raise NotImplementedError("M03-T5: write your code here")

In [ ]:
def envelope_evidence(records, row, origin, wrong_origin, before_s, after_s):
    """Envelope evidence for every record: distance, samples in the colleague's window, peak time and size.

    records      : obspy Stream of vertical records (any sampling rate) with the station coordinates in
                   tr.stats.sac.stla and tr.stats.sac.stlo; leave it unchanged (work on copies)
    row          : pandas Series or dict with Latitude and Longitude (degrees) of the epicenter
    origin       : obspy UTCDateTime, the true origin
    wrong_origin : obspy UTCDateTime, the colleague's origin
    before_s, after_s : float, seconds before and after each origin
    returns      : DataFrame with one row per record, nearest station first, and the columns
                   station      "NET.STA"
                   distance_km  epicentral distance from the row's latitude and longitude to the station, km
                   n_wrong      number of samples of the record from wrong_origin - before_s to
                                wrong_origin + after_s
                   t_peak_s     time of the envelope's peak from origin - before_s to origin + after_s, in seconds
                                after origin
                   peak_ratio   that peak divided by the median of the envelope over the first 20 s of the record
                   The envelope is that of the whole record after removing its mean, with no filter.
    """
    # Step 1: for every record, count its samples in the colleague's window.
    # Step 2: envelope of a copy of the record minus its mean; noise level; peak inside your window.
    # Step 3: distance to the station; one row per record; sort.
    # YOUR CODE HERE
    raise NotImplementedError("M03-T5: write your code here")

In [ ]:
# Build `evidence`: your envelope_evidence of records and trap_row, at the true origin (from your corrected_window)
# and the colleague's origin, 10 s before and 60 s after. Do not round or edit the table.
# YOUR CODE HERE
raise NotImplementedError("M03-T5: write your code here")
print(evidence.round(2).to_string())

In [ ]:
st.check("M03-T5", answer={"corrected_window": corrected_window, "envelope_evidence": envelope_evidence,
                           "evidence": evidence, "row": trap_row})

The evidence as a figure, with P and S travel times in a simple crust (6.0 and 3.5 km/s, straight rays):

In [ ]:
d_axis = np.linspace(0.0, RADIUS_KM, 101)
hypo_axis = np.hypot(d_axis, float(trap_row["Depth"]))
fig, ax = plt.subplots(figsize=(5.8, 4.0))
ax.plot(d_axis, hypo_axis / 6.0, color="0.5", lw=1.0, ls="--", label="P, 6.0 km/s")
ax.plot(d_axis, hypo_axis / 3.5, color="0.2", lw=1.0, label="S, 3.5 km/s")
dots = ax.scatter(evidence["distance_km"], evidence["t_peak_s"], c=np.log10(evidence["peak_ratio"]), s=28,
                  cmap="viridis", edgecolors="black", linewidths=0.4, zorder=3)
fig.colorbar(dots, ax=ax, label="log10 of the peak-to-noise ratio")
ax.set_xlabel("Epicentral distance (km)")
ax.set_ylabel("Envelope peak, s after the origin")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(Path(FIGURE_DIR) / "m03_envelope_evidence.png", dpi=150)
print(f"stations: {len(evidence)}; samples in the colleague's window: {int(evidence['n_wrong'].sum())}; "
      f"smallest peak-to-noise ratio {evidence['peak_ratio'].min():.0f}")

**Explain:** What went wrong in the colleague's cell, and why did it run without an error?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M03")

### Going further (optional, not checked)

1. Fetch `suncheon2019`, set `DATASET`, `EVENT_ID`, `EPICENTER` and `DEPTH_KM` to one of its events and run
   Parts 3-6 again. Which bands does the Suncheon area have?
2. Compute the Wood-Anderson magnitude of the other five Sangju events.

**Read more:** Ahn, Kang and Jung (2024), the station metadata; Havskov and Alguacil (2016), instruments and
responses; Bormann and Dewey (2014), magnitude formulas.

Next: M10 works with these seismograms.